# CONSORT prototype rebuild — 5-fold CV copy (2026-10-28)

The same pipeline as `CONSORT_Prototype_Rebuild_1014.ipynb`, run **once per fold** with that fold's training articles.

**Folds** (`data/cv5_1028/`): the 50 articles form 5 groups of 10. G0 is the original test set, G1 the original validation set, and G2–G4 the original 30 train articles, shuffled with seed 42. Fold *k* uses test = G*k*, validation = G*k+1*, and train = the other three groups (30 / 10 / 10). Every article is tested exactly once across the 5 folds.

| fold | train | validation | test |
|---|---|---|---|
| 0 | G2 G3 G4 | G1 | G0 (= the original split; prototype already built: `consort_prototype_rebuild_v3_1014.json`) |
| 1 | G0 G3 G4 | G2 | G1 |
| 2 | G0 G1 G4 | G3 | G2 |
| 3 | G0 G1 G2 | G4 | G3 |
| 4 | G1 G2 G3 | G0 | G4 |

**What changes vs the 1014 notebook:**
- `FOLD` in the configuration cell picks the split file, the output folder and the frozen prototype path.
- V0 and V1 use no corpus data, so they are **reused** from the 1014 run (`REUSE_V0_V1`). V2, the training extraction, the error analysis, V3, the freeze and the held-out test all run per fold, on that fold's train articles only.
- The router decisions come from the same Step 1 router cache as before. The router uses no prototype and no gold, so it is the same in every fold.

**Cost per fold:** about 117 calls for V2, ~150 for the training extraction, ~150 for the error analysis, ~40 for V3 and ~100 for the held-out test.

# CONSORT prototype rebuild (train-only), 2026-10-14

**Why.** The prototype the pipeline has used (`consort_prototype_pass3_error_grounded_v1.json`) was built from an older split. 100 of its 177 boundary-rule examples are gold-labelled sentences from **validation (17) and test (83)** articles of the current split, and 48 cue phrases occur only in validation/test text. This notebook rebuilds the prototype from scratch:
- the documents (guideline, checklist, literature) are used in V0/V1;
- **training articles only** are used for every data-driven stage;
- the output has **the same JSON structure** as pass3, so it is a drop-in replacement for the v26 / v34 pipelines.

| step | stage | inputs | LLM calls (approx.) |
|---|---|---|---|
| 1 | **V0 — canonical initialization** | annotation guideline + CONSORT checklist | 37 items + 1 rules + 1 global |
| 2 | **V1 — literature enrichment** | + CONSORT 2010 E&E + CONSORT-TM paper | ~15 chunks + 37 items + 1 rules |
| 3 | **split verification** | `split_40_10.json` (the pipeline's split): 30 train / 10 validation / 10 test | – |
| 4 | **V2 — gold-boundary audit** | train gold positives + counterexamples per item and per boundary rule | 37 + ~80 |
| 5 | **training extraction** | V2 on the 30 train articles: v26 section extractor | ~150 |
| 6 | **training error analysis** | category + ≤20-word reason for every FP/FN | ~150 |
| 7 | **V3 — empirical refinement** | training errors only | 37 + rules + 1 |
| 8 | **freeze V3** | leakage audit + hash + freeze record | – |
| 9 | **held-out test** | frozen V3 on the 10 test articles (diagnostic; never updates V3) | ~50 + ~50 |

**Leakage rules enforced in code:**
- every gold-reading stage asserts it only sees train articles;
- boundary-rule examples are chosen **by ID** from train candidates and the exact text is inserted, so the LLM never writes example text;
- the freeze step audits the prototype for any validation/test sentence and for cue phrases attested only outside train.

**Same extractor as the pipeline.** Step 5 is the v26 Step-2 extractor exactly: the same `render_fixed_section_extraction_spec` / `render_mutable_section_prototype_memory` and the same DSPy signature, one call per (article, section). Paragraph routing and the article memory come from the Step 1 cache, because the router doesn't use the prototype. Set `ROUTING = "gold"` for oracle routing.

All stages are cached under `OUTPUT_DIR/cache/`, so interrupted runs resume, and a revised prototype never reuses stale extraction calls.


## 0. Configuration

In [15]:
import os, sys, json, getpass
from pathlib import Path
import pandas as pd
import dspy

sys.path.insert(0, str(Path.cwd()))
import importlib
import consort_prototype_rebuild as R
importlib.reload(R)

# ---- [cv5] the fold this run builds the prototype for (1-4; fold 0 = the original split, already built) ----
FOLD = 4
assert FOLD in (1, 2, 3, 4), "fold 0 is the original split: its prototype is consort_prototype_rebuild_v3_1014.json"

BASE_DIR = Path('../data')
OUTPUT_DIR = BASE_DIR / 'outputs_supervised_cv5_1028' / f'fold{FOLD}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GUIDELINE_PATH = BASE_DIR / 'annotation_guideline_v1.txt'
CHECKLIST_PATH = BASE_DIR / 'CONSORT_2010_Checklist.csv'
CONSORT_EANDE_PATH = BASE_DIR / 'consort_2010_explanation.txt'
CONSORT_TM_PATH = BASE_DIR / 'consort_tm_paper.txt'
DATA_PATH = BASE_DIR / 'sentences.csv'
PARAGRAPHS_PATH = BASE_DIR / 'paragraphs_with_sections.csv'

cfg = R.RebuildConfig(
    base_dir=BASE_DIR, output_dir=OUTPUT_DIR,
    guideline_path=GUIDELINE_PATH, checklist_path=CHECKLIST_PATH,
    eande_path=CONSORT_EANDE_PATH, consort_tm_path=CONSORT_TM_PATH,
    data_path=DATA_PATH, paragraphs_path=PARAGRAPHS_PATH,
    split_path=BASE_DIR / 'cv5_1028' / f'split_fold{FOLD}.json',                 # [cv5] this fold's split
    structured_guideline_path=BASE_DIR / 'annotation_guideline_v3_structured.json',  # rendered by the v26 extractor spec
    step1_cache=Path('../output_and_result/v29_pseudo_label_enriched_classifier_terra/llm_pseudo_labels'),
    model='gpt-5.6-terra', reasoning_effort='medium', n_workers=4,
)
ROUTING = 'predicted'          # 'predicted' (Step 1 router, as in the pipeline) or 'gold' (oracle)
FINAL_PROTOTYPE_PATH = Path('../output_and_result/cv5_1028') / f'fold{FOLD}' / 'consort_prototype_rebuild_v3.json'
FINAL_PROTOTYPE_PATH.parent.mkdir(parents=True, exist_ok=True)
# [cv5] V0 and V1 read no corpus data (guideline, checklist, literature only), so every fold reuses the
# V0/V1 built in the original run. Set False to regenerate them (~90 extra LLM calls).
REUSE_V0_V1 = True
V0V1_SOURCE = BASE_DIR / 'outputs_supervised_1014'
print(f'FOLD {FOLD} | split', cfg.split_path, '| output', OUTPUT_DIR, '| prototype ->', FINAL_PROTOTYPE_PATH)

for k in ['guideline_path', 'checklist_path', 'eande_path', 'consort_tm_path', 'data_path', 'paragraphs_path',
          'split_path', 'structured_guideline_path', 'step1_cache']:
    p = getattr(cfg, k)
    print(f"{k:28s} {p}  {'✓' if Path(p).exists() else 'MISSING'}")

FOLD 4 | split ../data/cv5_1028/split_fold4.json | output ../data/outputs_supervised_cv5_1028/fold4 | prototype -> ../output_and_result/cv5_1028/fold4/consort_prototype_rebuild_v3.json
guideline_path               ../data/annotation_guideline_v1.txt  ✓
checklist_path               ../data/CONSORT_2010_Checklist.csv  ✓
eande_path                   ../data/consort_2010_explanation.txt  ✓
consort_tm_path              ../data/consort_tm_paper.txt  ✓
data_path                    ../data/sentences.csv  ✓
paragraphs_path              ../data/paragraphs_with_sections.csv  ✓
split_path                   ../data/cv5_1028/split_fold4.json  ✓
structured_guideline_path    ../data/annotation_guideline_v3_structured.json  ✓
step1_cache                  ../output_and_result/v29_pseudo_label_enriched_classifier_terra/llm_pseudo_labels  ✓


**API key.** It's read from the `OPENAI_API_KEY` environment variable, or typed when prompted. Never paste a key into a notebook cell: it ends up saved in the `.ipynb` file. The LM is configured globally for DSPy, and the extractor uses the same settings as the pipeline.

In [16]:
api_key = os.environ.get('OPENAI_API_KEY') or getpass.getpass('OpenAI API key: ')
llm = R.JsonLLM(cfg.model, cfg.reasoning_effort, api_key=api_key)
dspy.configure(lm=llm.lm)
print('LLM:', cfg.model, '| reasoning:', cfg.reasoning_effort)

LLM: gpt-5.6-terra | reasoning: medium


## 1–2 run after the split check; step 3 (split verification) runs first

The split is loaded before V0 so that every later stage can assert it only reads training articles. V0 and V1 read no corpus data at all.

In [17]:
corpus = R.load_corpus(cfg)
display(R.verify_split(corpus))
guideline_obj = R.load_annotation_guideline(cfg.structured_guideline_path)
missing_step1 = sorted(set(corpus['frame'].article) - set(corpus['records']))
print('Articles without a Step 1 record (needed for predicted routing):', missing_step1 or 'none')

,articles,sentences,gold pairs
split,,,
train,30,6406,3000
validation,10,2177,1014
test,10,2126,1041


Articles without a Step 1 record (needed for predicted routing): none


## 1. V0: canonical initialization

Inputs: the annotation guideline and the CONSORT checklist only. One call per item produces `must_have`, `not_if`, positive/negative cues, `section_prior` and `high_risk_confusions`. One call then writes pairwise boundary rules for confusable pairs, and one writes global rules. **No corpus data.**

In [18]:
if REUSE_V0_V1:   # [cv5] fold-independent
    v0 = json.loads((V0V1_SOURCE / 'v0_prototype.json').read_text())
else:
    v0 = R.stage_v0(cfg, llm)
R.save_prototype(v0, OUTPUT_DIR / 'v0_prototype.json')
print(f"V0: {v0['metadata']['num_items']} items, {v0['metadata']['num_pairwise_boundary_rules']} boundary rules, "
      f"{len(v0['global_rules'])} global rules")
ITEM = '17b'
print(json.dumps(v0['item_prototypes'][ITEM], indent=1, ensure_ascii=False))

V0: 37 items, 109 boundary rules, 20 global rules
{
 "must_have": [
  "Explicitly reports both an absolute effect size and a relative effect size for a binary outcome; the paired estimates may appear in linked sentence fragments."
 ],
 "not_if": [
  "Only an absolute or only a relative effect size is reported.",
  "The outcome is non-binary; assign 17a when reporting results and estimated effects for primary or secondary non-binary outcomes.",
  "The sentence reports group event counts or percentages without both absolute and relative effect estimates; this is generally 17a when it reports a primary or secondary outcome.",
  "The sentence only reports statistical significance, a P value, or a confidence interval without paired absolute and relative effects.",
  "The sentence reports harms or adverse events without the paired absolute and relative effect presentation; consider item 19."
 ],
 "positive_cues": [
  "absolute risk difference",
  "relative risk reduction",
  "risk ratio and 

## 2. V1: literature enrichment

The two papers are read in chunks; each chunk call returns notes for any item. Each item is then revised with its notes, and the boundary rules with the confusion notes. Every revision records its source chunks in `metadata.provenance`.

**Missing file:** `consort_tm_paper.txt` is not in `data/`. Add the CONSORT-TM paper as plain text, or pass `require_consort_tm=False`; the metadata then records that only E&E was used.

In [19]:
if REUSE_V0_V1:   # [cv5] fold-independent
    v1 = json.loads((V0V1_SOURCE / 'v1_prototype.json').read_text())
else:
    v1 = R.stage_v1(cfg, llm, v0, require_consort_tm=True)
R.save_prototype(v1, OUTPUT_DIR / 'v1_prototype.json')
print('Sources:', v1['metadata']['source_files'])
print('Notes per item (top 10):', dict(sorted(v1['metadata']['v1_notes_per_item'].items(), key=lambda x: -x[1])[:10]))
for name, p in [('V0', v0), ('V1', v1)]:
    print(f"\n=== {name} {ITEM} ===")
    print(json.dumps({k: p['item_prototypes'][ITEM][k] for k in ['must_have', 'not_if']}, indent=1, ensure_ascii=False))

Sources: ['annotation_guideline_v1.txt', 'CONSORT_2010_Checklist.csv', 'consort_2010_explanation.txt', 'consort_tm_paper.txt']
Notes per item (top 10): {'1b': 12, '21': 12, '13a': 11, '3a': 10, '11a': 10, '16': 10, '12b': 10, '7b': 9, '6a': 9, '13b': 9}

=== V0 17b ===
{
 "must_have": [
  "Explicitly reports both an absolute effect size and a relative effect size for a binary outcome; the paired estimates may appear in linked sentence fragments."
 ],
 "not_if": [
  "Only an absolute or only a relative effect size is reported.",
  "The outcome is non-binary; assign 17a when reporting results and estimated effects for primary or secondary non-binary outcomes.",
  "The sentence reports group event counts or percentages without both absolute and relative effect estimates; this is generally 17a when it reports a primary or secondary outcome.",
  "The sentence only reports statistical significance, a P value, or a confidence interval without paired absolute and relative effects.",
  "The sen

## 3. Split verification

This repeats the check from above, to confirm the exact article-level split before any stage reads gold labels:
- **30 train articles:** the only gold used from here on;
- **10 validation articles:** untouched here, reserved for the downstream pipeline's tuning;
- **10 test articles:** step 9, once.

In [20]:
split_table = R.verify_split(corpus)
display(split_table)
assert set(R.train_rows(corpus).article) == set(corpus['split']['train'])
R.save_json_file({k: sorted(v) for k, v in corpus['split'].items() if isinstance(v, list)}, OUTPUT_DIR / 'split_manifest.json')

,articles,sentences,gold pairs
split,,,
train,30,6406,3000
validation,10,2177,1014
test,10,2126,1041


## 4. V2: gold-boundary audit (train gold only)

- **Per item:** up to 16 gold positives and 12 counterexamples from train. Counterexamples are sentences gold-labelled with one of the item's `high_risk_confusions`, or other labelled sentences in its section. The LLM flags rules that are too strict or too permissive, or cues treated as requirements, and revises the item.
- **Per boundary rule:** up to 6 train sentences annotated with each side only. The LLM revises the rule and **selects up to 3 example IDs**; the code inserts those sentences verbatim, with the article and sid recorded in `metadata.example_sources`.

In [21]:
v2 = R.stage_v2(cfg, llm, v1, corpus)
R.save_prototype(v2, OUTPUT_DIR / 'v2_prototype.json')
audit = pd.read_csv(OUTPUT_DIR / 'v2_gold_boundary_audit.csv')
display(audit[audit['field'] != 'pairwise_rule'].groupby(['field', 'issue']).size().rename('findings').to_frame())
display(audit[audit['item'] == ITEM])
RULE = '17a__17b'
print(json.dumps(v2['pairwise_boundary_rules'].get(RULE, {}), indent=1, ensure_ascii=False)[:2000])

V2 item audit:   0%|          | 0/37 [00:00<?, ?it/s]

V2 rule audit:   0%|          | 0/113 [00:00<?, ?it/s]

findings
field                issue                       
high_risk_confusions ambiguous                 30
                     too_permissive             4
must_have            ambiguous                  5
                     cue_as_requirement         6
                     too_permissive             3
                     too_strict                74
negative_cues        ambiguous                 10
                     cue_as_requirement        12
                     too_permissive             6
                     too_strict                 6
not_if               ambiguous                 15
                     cue_as_requirement         1
                     too_permissive             9
                     too_strict                70
positive_cues        ambiguous                  4
                     cue_as_requirement        32
                     too_permissive             3
                     too_strict                 1
section_prior        ambiguous                 10
                     too_permissive             1
                     too_strict                17

,item,field,issue,finding,evidence_ids,change
224,17b,must_have,too_strict,Requiring paired absolute and relative estimat...,"['P1', 'P3', 'P4', 'P7', 'P9', 'P10', 'P12', '...",Define the item operationally as reporting a r...
225,17b,not_if,too_strict,"Excluding single effect measures, significance...","['P1', 'P3', 'P4', 'P7', 'P10', 'P12', 'P13', ...",Do not reject solely because only one effect m...
226,17b,not_if,too_strict,The blanket exclusion of harms and adverse-eve...,"['P2', 'P5', 'P6', 'P8', 'P15']",Permit co-assignment with item 19 when a safet...
227,17b,not_if,too_strict,Excluding hazard-ratio-only reporting conflict...,['P4'],Do not use the presence of a hazard ratio as a...
228,17b,positive_cues,cue_as_requirement,The cue set is narrowly centered on paired abs...,"['P1', 'P2', 'P7', 'P9', 'P12', 'P13']",Use effect-pairing language only as optional e...
229,17b,negative_cues,too_strict,Several negative cues incorrectly reject gold-...,"['P2', 'P3', 'P4', 'P5', 'P6', 'P8', 'P10', 'P...","Remove single relative estimates, event freque..."
230,17b,not_if,too_permissive,"After broadening beyond paired estimates, bina...","['N6', 'N8', 'N9', 'N10', 'N12']",Require that the result concern a randomized-g...
231,17b,not_if,ambiguous,A simple group-proportion comparison is presen...,"['P3', 'P7', 'P10', 'P12', 'P14', 'N10']",Use outcome role and analysis purpose as the p...
232,17b,section_prior,ambiguous,"All supplied gold positives occur in Results, ...","['P2', 'P5', 'P6', 'P15']",Keep Results as a strong prior and explicitly ...
233,17b,high_risk_confusions,ambiguous,The principal observed confusions are 17a for ...,"['N1', 'N2', 'N3', 'N4', 'N5', 'N6', 'N8', 'N9...","Prioritize 17a, 18, and 19 as high-risk confus..."


{
 "items": [
  "17a",
  "17b"
 ],
 "observed_confusions": 0,
 "rule": "Use 17a for outcome findings and ancillary results; use 17b for between-group binary endpoint event or response rates.",
 "choose_first_when": "general, continuous, time-based, or ancillary findings",
 "choose_second_when": "binary endpoint rates across groups",
 "do_not": "classify every paired proportion as 17b",
 "examples": [
  {
   "section": "Results",
   "item": "17a",
   "sentence": "For patients not taking antibiotics at enrolment, antibiotic prescribing was similar (P=0.27) for both arms after enrolment: 31/50 (62%) in the double dose and 33/45 (73%) in the standard dose arm.",
   "source": "gold_training_positive"
  },
  {
   "section": "Results",
   "item": "17b",
   "sentence": "There were no intraoperative RBC transfusions in the intervention group compared with 5 in the usual care group (P = 0.014) and a significant reduction in the number of total perioperative ABT events in group 1 (5/40, 12.5 %) c

## 5. Training extraction with V2

These are the v26 Step-2 section extractors, rendered from the V2 prototype and run on the 30 train articles. Metrics are pair-level micro P/R/F1 over (sentence, item), with 1a/1b/2a excluded as in the pipeline.

This is the **extractor stage alone**: no Step-1 verifier or recovery. So it measures the prototype, not the full pipeline.

In [22]:
train_pairs_v2 = R.run_extraction(cfg, v2, corpus, 'train', guideline_obj, routing=ROUTING, tag='train_v2')
train_summary, train_items, train_errors = R.evaluate_pairs(corpus, 'train', train_pairs_v2)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in train_summary.items()})
display(train_items.sort_values('support', ascending=False).round(3))

extract [train]:   0%|          | 0/150 [00:00<?, ?it/s]

{'split': 'train', 'P': 0.681, 'R': 0.759, 'F1': 0.718, 'tp': 2277, 'fp': 1066, 'fn': 723}


,item,support,tp,fp,fn,P,R,F1
24,17a,404,204,42,200,0.829,0.505,0.628
6,6a,393,321,54,72,0.856,0.817,0.836
30,22,375,319,207,56,0.606,0.851,0.708
26,18,161,123,169,38,0.421,0.764,0.543
5,5,155,133,47,22,0.739,0.858,0.794
16,12a,147,113,54,34,0.677,0.769,0.720
28,20,120,101,58,19,0.635,0.842,0.724
22,15,114,102,11,12,0.903,0.895,0.899
27,19,107,100,63,7,0.613,0.935,0.741
3,4a,100,93,10,7,0.903,0.930,0.916


## 6. Training error analysis

Every FP and FN gets a category from a fixed list, a reason of at most 20 words, and the competing item. Errors are batched per (article, section) and shown with the sentence, its neighbours, and the gold and predicted items.

In [23]:
train_err = R.run_error_analysis(cfg, llm, corpus, train_errors, train_pairs_v2, v2, tag='train_v2')
train_err.to_csv(OUTPUT_DIR / 'train_error_analysis.csv', index=False)
display(train_err.groupby(['status', 'category']).size().rename('n').reset_index().sort_values('n', ascending=False).head(15))
display(train_err[train_err['item'] == ITEM][['status', 'category', 'confused_with', 'reason', 'text']].head(12))

error analysis [train_v2]:   0%|          | 0/139 [00:00<?, ?it/s]

,status,category,n
9,FP,ANNOTATION_POLICY_MISMATCH,336
17,FP,WRONG_OUTCOME_TYPE,230
5,FN,MULTILABEL_OMISSION,178
2,FN,CONTEXT_DEPENDENT,170
16,FP,TOPICAL_BUT_NOT_EVIDENCE,125
12,FP,INSUFFICIENT_SPECIFICITY,111
14,FP,MULTILABEL_BOUNDARY,109
1,FN,CAPTION_OR_TABLE_EVIDENCE,100
3,FN,IMPLICIT_OR_INDIRECT_EVIDENCE,76
11,FP,CONTEXT_MISREAD,71


,status,category,confused_with,reason,text
41,FP,ANNOTATION_POLICY_MISMATCH,17a,Group-specific deterioration is a trial outcom...,Nine participants (six children and three adul...
83,FP,ANNOTATION_POLICY_MISMATCH,17a,"Although binary, annotators retained this prim...",Eight of 12 patients (67%) in group 1 and only...
180,FP,CAPTION_OR_TABLE_MISREAD,none,This figure fragment names a mortality display...,Rate of death in the two study groups up to 28...
181,FP,ANNOTATION_POLICY_MISMATCH,17a,Gold routed the mortality hazard-ratio result ...,Survival analysis of the primary outcome (figu...
182,FP,ANNOTATION_POLICY_MISMATCH,17a,Gold routed the absolute mortality-risk result...,Salbutamol resulted in a 10·9% (95% CI 1·0–20·...
183,FP,ANNOTATION_POLICY_MISMATCH,17a,Gold routed the number-needed-to-harm mortalit...,One additional death occurred for every 9·2 (9...
184,FP,ANNOTATION_POLICY_MISMATCH,17a,Gold routed mortality comparison results to 17...,The number of deaths before discharge from eit...
185,FP,ANNOTATION_POLICY_MISMATCH,17a,Gold routed absolute ICU and hospital mortalit...,We noted an 8·4% (95% CI −1·7 to 18·3) absolut...
186,FP,ANNOTATION_POLICY_MISMATCH,17a,Gold routed the primary-outcome risk ratio to ...,The RR for the primary outcome at this time wa...
215,FP,ANNOTATION_POLICY_MISMATCH,17a,Binary primary-outcome results were annotated ...,The initial results after data lock and unblin...


## 7. V3: empirical refinement (training errors only)

- **Per item:** revised from its training error patterns; only generalizable, rule-level changes are accepted. The counts fields (`error_count`, `omission_count`, …) are set from the training errors.
- **Per boundary rule:** `observed_confusions` = same-sentence confusions on train (gold *a*, predicted *b*). Rules with confusions are revised and get up to 2 error-log examples **from train**. New rules are created for unruled pairs with ≥ 5 confusions.
- **Global rules:** revised from the top error categories.

Every accepted change goes to `v3_change_log.csv`.

In [24]:
v3 = R.stage_v3(cfg, llm, v2, corpus, train_err)
R.save_prototype(v3, OUTPUT_DIR / 'v3_prototype.json')
display(pd.read_csv(OUTPUT_DIR / 'v3_change_log.csv').head(30))
for name, p in [('V2', v2), ('V3', v3)]:
    print(f"\n=== {name} {ITEM} ===")
    print(json.dumps({k: p['item_prototypes'][ITEM][k] for k in ['must_have', 'not_if', 'error_count']}, indent=1, ensure_ascii=False))

V3 items:   0%|          | 0/37 [00:00<?, ?it/s]

V3 rules:   0%|          | 0/36 [00:00<?, ?it/s]

,target,field,change,pattern,n_errors
0,3b,must_have,Clarified that implemented post-commencement c...,False positive caused by treating a later allo...,1.0
1,3b,must_have,Clarified that implementation and post-commenc...,Context-dependent false negatives where the ch...,2.0
2,3b,not_if,Distinguished an original allocation descripti...,Confusion between changed methods after commen...,2.0
3,3b,positive_cues,Added generic cues for later allocation modifi...,Missed post-commencement method changes signal...,3.0
4,2b,must_have,Allow prospective conditional predictions of a...,Missed indirect or context-dependent hypothese...,3.0
5,2b,not_if,Explicitly exclude mechanistic or observationa...,False positives from biological or correlation...,1.0
6,2b,not_if,Clarify that a statement that a study was esta...,False positives from broad descriptions of stu...,1.0
7,2b,section_prior,Expand location guidance beyond Abstract and I...,Missed explicit aims located outside the expec...,2.0
8,2b,positive_cues,Add cues for conditional prospective benefit p...,Missed prospective hypotheses expressed indire...,3.0
9,3a,must_have,Clarified that qualifying evidence must explic...,Prevents false positives from vague structural...,4.0



=== V2 17b ===
{
 "must_have": [
  "Explicitly reports a randomized-group binary clinical or safety outcome result, including in a linked results table or fragment.",
  "The result may be conveyed by event counts, proportions, responder status, an absolute estimate, a relative estimate, NNT/NNH, or a comparative conclusion.",
  "Paired absolute and relative effects with confidence intervals are strong supporting evidence, but are not required."
 ],
 "not_if": [
  "The result concerns a continuous or otherwise non-binary outcome; assign 17a when it reports a primary or secondary outcome result.",
  "The statement is a predictor, prognostic, or observational association rather than a randomized-group outcome comparison; consider item 18 when it is an additional analysis.",
  "The result is an exploratory subgroup, cohort, or period-effect analysis rather than reporting the trial outcome result; consider item 18.",
  "The text is a baseline or descriptive characteristics table rather tha

### Optional: check V3 on train before freezing

Re-run the extractor with V3 on the train articles (~150 calls). An improvement here is expected, since V3 was fitted to these errors. Treat it as a sanity check, not an estimate.

In [25]:
RUN_V3_TRAIN_CHECK = False
if RUN_V3_TRAIN_CHECK:
    pairs_v3_train = R.run_extraction(cfg, v3, corpus, 'train', guideline_obj, routing=ROUTING, tag='train_v3')
    display(pd.DataFrame([R.evaluate_pairs(corpus, 'train', train_pairs_v2)[0] | {'prototype': 'V2'},
                          R.evaluate_pairs(corpus, 'train', pairs_v3_train)[0] | {'prototype': 'V3'}]).set_index('prototype').round(3))

## 8. Freeze V3

The leakage audit must pass before freezing:
- no validation/test sentence may appear in prompt-visible content;
- every multi-word cue must be attested in some train sentence.

Cues that fail the second check are removed and recorded. The frozen file goes to `output_and_result/consort_prototype_rebuild_v3_1014.json`, for the pipeline.

In [26]:
audit_v3 = R.leakage_audit(v3, corpus)
print('held-out sentences in prototype:', len(audit_v3['heldout_sentences_in_prototype']),
      '| cues attested only in held-out text:', audit_v3['cues_only_in_heldout_text'])
if audit_v3['cues_only_in_heldout_text']:
    v3 = R.drop_heldout_cues(v3, audit_v3)
    v3['metadata']['removed_heldout_cues'] = [list(x) for x in audit_v3['cues_only_in_heldout_text']]
    audit_v3 = R.leakage_audit(v3, corpus)
assert audit_v3['clean'], audit_v3
v3 = R.finalize_metadata(v3, corpus)
v3['metadata'].update({'frozen': True, 'frozen_at': str(pd.Timestamp.now()), 'routing_during_training': ROUTING,
                       'model': cfg.model, 'split_file': str(cfg.split_path)})
R.save_prototype(v3, OUTPUT_DIR / 'v3_prototype_frozen.json')
assert not FINAL_PROTOTYPE_PATH.exists() or json.loads(FINAL_PROTOTYPE_PATH.read_text())['metadata']['prototype_hash'] == v3['metadata']['prototype_hash'], \
    f'{FINAL_PROTOTYPE_PATH} exists with a different prototype; choose a new name'
R.save_prototype(v3, FINAL_PROTOTYPE_PATH)
R.save_json_file({'prototype': str(FINAL_PROTOTYPE_PATH), 'hash': v3['metadata']['prototype_hash'],
                  'frozen_at': v3['metadata']['frozen_at']}, OUTPUT_DIR / 'freeze_record.json')
print('Frozen V3:', FINAL_PROTOTYPE_PATH, '| hash', v3['metadata']['prototype_hash'],
      f"| {v3['metadata']['num_items']} items, {v3['metadata']['num_pairwise_boundary_rules']} rules")

held-out sentences in prototype: 0 | cues attested only in held-out text: [('3b', 'planned interim analysis'), ('3b', 'stopping rule'), ('4a', 'second randomisation'), ('6a', 'clinical score'), ('6a', 'treatment administration'), ('8b', 'variable block sizes'), ('12b', 'covariate adjustment'), ('13a', 'randomly assigned participants'), ('14a', 'enrolment period'), ('16', 'included in analysis'), ('16', 'treatment duration'), ('17a', 'between-group difference'), ('18', 'subgroup analysis showed'), ('20', 'wide confidence intervals'), ('24', 'ethics committee approval'), ('25', 'funding sources')]
Frozen V3: ../output_and_result/cv5_1028/fold4/consort_prototype_rebuild_v3.json | hash 710a37fa077f6735 | 37 items, 114 rules


## 9. Held-out test (once)

Frozen V3 runs on the 10 test articles with the same extractor and routing. The error analysis is **diagnostic only**: nothing below feeds back into the prototype.

A lock file records the prototype hash at the first test run. Re-running with a different prototype stops with an error.

In [27]:
lock = OUTPUT_DIR / 'test_lock.json'
if lock.exists():
    assert json.loads(lock.read_text())['hash'] == v3['metadata']['prototype_hash'], 'test already evaluated with another prototype'
else:
    R.save_json_file({'hash': v3['metadata']['prototype_hash'], 'time': str(pd.Timestamp.now())}, lock)

test_pairs = R.run_extraction(cfg, v3, corpus, 'test', guideline_obj, routing=ROUTING, tag='test_v3')
test_summary, test_items, test_errors = R.evaluate_pairs(corpus, 'test', test_pairs)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in test_summary.items()})
display(test_items.sort_values('support', ascending=False).round(3))
test_err = R.run_error_analysis(cfg, llm, corpus, test_errors, test_pairs, v3, tag='test_v3')
test_err.to_csv(OUTPUT_DIR / 'test_error_analysis.csv', index=False)
display(test_err.groupby(['status', 'category']).size().rename('n').reset_index().sort_values('n', ascending=False).head(15))

extract [test]:   0%|          | 0/50 [00:00<?, ?it/s]

{'split': 'test', 'P': 0.699, 'R': 0.768, 'F1': 0.732, 'tp': 799, 'fp': 344, 'fn': 242}


,item,support,tp,fp,fn,P,R,F1
24,17a,177,116,10,61,0.921,0.655,0.766
6,6a,155,125,21,30,0.856,0.806,0.831
30,22,133,114,45,19,0.717,0.857,0.781
16,12a,66,50,15,16,0.769,0.758,0.763
5,5,61,50,14,11,0.781,0.820,0.800
27,19,52,51,40,1,0.560,0.981,0.713
22,15,39,32,3,7,0.914,0.821,0.865
3,4a,33,33,2,0,0.943,1.000,0.971
25,17b,30,15,22,15,0.405,0.500,0.448
28,20,29,24,21,5,0.533,0.828,0.649


error analysis [test_v3]:   0%|          | 0/46 [00:00<?, ?it/s]

,status,category,n
9,FP,ANNOTATION_POLICY_MISMATCH,97
2,FN,CONTEXT_DEPENDENT,70
17,FP,WRONG_OUTCOME_TYPE,70
5,FN,MULTILABEL_OMISSION,52
12,FP,INSUFFICIENT_SPECIFICITY,43
14,FP,MULTILABEL_BOUNDARY,42
1,FN,CAPTION_OR_TABLE_EVIDENCE,40
16,FP,TOPICAL_BUT_NOT_EVIDENCE,32
11,FP,CONTEXT_MISREAD,23
0,FN,ANNOTATION_POLICY_MISMATCH,21


## 10. Experiment record and next steps

In [28]:
record = {'split': str(cfg.split_path), 'routing': ROUTING, 'model': cfg.model,
          'prototypes': {v: str(OUTPUT_DIR / f'{v}_prototype.json') for v in ['v0', 'v1', 'v2', 'v3']},
          'frozen': str(FINAL_PROTOTYPE_PATH), 'hash': v3['metadata']['prototype_hash'],
          'train_v2_extraction': train_summary, 'test_v3_extraction': test_summary}
R.save_json_file(record, OUTPUT_DIR / 'experiment_record.json')
print(json.dumps(record, indent=1, default=str))

{
 "split": "../data/cv5_1028/split_fold4.json",
 "routing": "predicted",
 "model": "gpt-5.6-terra",
 "prototypes": {
  "v0": "../data/outputs_supervised_cv5_1028/fold4/v0_prototype.json",
  "v1": "../data/outputs_supervised_cv5_1028/fold4/v1_prototype.json",
  "v2": "../data/outputs_supervised_cv5_1028/fold4/v2_prototype.json",
  "v3": "../data/outputs_supervised_cv5_1028/fold4/v3_prototype.json"
 },
 "frozen": "../output_and_result/cv5_1028/fold4/consort_prototype_rebuild_v3.json",
 "hash": "710a37fa077f6735",
 "train_v2_extraction": {
  "split": "train",
  "P": 0.6811247382590487,
  "R": 0.759,
  "F1": 0.7179568027747123,
  "tp": 2277,
  "fp": 1066,
  "fn": 723
 },
 "test_v3_extraction": {
  "split": "test",
  "P": 0.6990376202974629,
  "R": 0.7675312199807877,
  "F1": 0.7316849816849818,
  "tp": 799,
  "fp": 344,
  "fn": 242
 }
}


**Next:** run `prompt_optimize_1028_v34_cv5_fold.ipynb` with the same `FOLD`. It reads the prototype frozen here (`output_and_result/cv5_1028/fold<k>/consort_prototype_rebuild_v3.json`).

The test number in step 9 here covers the **extractor alone** on this fold's test articles, so it is not the pipeline's final number.